In [ ]:
from pathlib import Path
from IPython.display import Image, SVG, display
import getpass
import os
import re
import time

# Works when this notebook starts in notebooks/, the repository root, or Gadi.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))]
repo_root = next((path for path in candidates
                  if (path / "exercises/CMakeLists.txt").is_file()
                  and (path / "job-script").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the intro-to-cuda checkout.")
os.chdir(repo_root)
job_dir = repo_root / "job-script"
username = getpass.getuser()


def show_job_output(job_name, job_id):
    job_number = job_id.split(".")[0]
    log = job_dir / f"{job_name}.o{job_number}"
    if log.is_file():
        print(log.read_text())
    else:
        print(f"Output is not available yet: {log.name}. Check qstat, then rerun this cell.")


def outputs_ready(paths, submitted_at):
    if all(path.is_file() and path.stat().st_mtime >= submitted_at for path in paths):
        return True
    print("New results are not available yet. Wait for this job to finish, then rerun this cell.")
    return False

print(f"Repository: {repo_root}")

# Asynchronous CUDA Calls


> **Overview**
>
> 
>  * **Time:** 60 min
> 
>  1. Learn how to use asynchronous CUDA APIs to improve performance.
>  1. Understand the concept of CUDA streams and how they can be used to overlap memory transfers and kernel execution.

This section provides an example of how to use asynchronous CUDA APIs to improve the performance of a simple vector addition program.
By using CUDA streams, we can overlap memory transfers and kernel execution, leading to better GPU utilization.

## CUDA Streams


A stream is a sequence of operations that execute on the GPU in the order they are issued. Streams allow for concurrent execution of multiple operations, such
as memory transfers and kernel launches, which can significantly improve performance by overlapping computation and data transfer.

In [ ]:
# Streams Overview
display(Image(filename=str(repo_root / "docs/source/tutorial/figs/streams.png")))

Streams are particularly useful in scenarios where you want to perform multiple operations on the GPU without waiting for each operation to complete before
starting the next one. By using streams, you can maximize the utilization of the GPU and reduce idle time.

## Why streams result in more concurrency?


1. Streaming Multiprocessors (SMs)

    * GPUs contain multiple Streaming Multiprocessors (SMs).
    * Each SM can execute threads from different kernels concurrently, as long as there's enough space and resources (registers, shared memory, etc.).
    * So parallel kernel launches can utilize multiple SMs, allowing for more threads to be executed simultaneously.

2. Separate Engines for Compute and Memory Operations

    * GPUs have separate engines for memory operations (like `cudaMemcpyAsync`) and compute operations (like kernel launches).
    * This means that while one stream is executing a kernel, another stream can perform memory transfers without waiting for the first to finish.
    * This overlap allows for better utilization of the GPU's resources.


For efficient and proper use of streams, it's important to ensure that

* The GPU must support concurrent kernel execution.
* There must be no resource conflict (e.g., using different memory regions).
* Streams must be independent — no implicit synchronization (e.g., from using the default stream).


> **Important**
>
> Streams are not a replacement for parallelism; they are a way to manage concurrency on the GPU. You can have multiple streams executing concurrently, 
> but each stream's operations will still be executed in order.

## Main differences from the synchronous version


* Create and use a CUDA stream with `cudaStreamCreate()`.
* Use `cudaMemcpyAsync()` for asynchronous memory copies.
* Launch the kernel in the created stream.
* Synchronize the stream with `cudaStreamSynchronize()` to wait for completion.
* Cleanup the stream after use.



> **Important**
>
> `cudaMalloc` and `cudaFree` are still synchronous operations, meaning they block the host until the memory allocation or deallocation is complete.

```cpp
#include <stdio.h>
#include <cuda_runtime.h>

// Kernel 1: Adds 1 to each element
__global__ void kernel1(float* data, int n) 
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n)
        data[idx] += 1.0f;
}

// Kernel 2: Multiplies each element by 2
__global__ void kernel2(float* data, int n) 
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n)
        data[idx] *= 2.0f;
}

int main() 
{
    const int n1 = 512;
    const int n2 = 512;
    const int size1 = n1 * sizeof(float);
    const int size2 = n2 * sizeof(float);

    float *h_data1 = (float*)malloc(size1);
    float *h_data2 = (float*)malloc(size2);

    // Initialize host data
    for (int i = 0; i < n1; i++) h_data1[i] = i;
    for (int i = 0; i < n2; i++) h_data2[i] = i * 10;

    float *d_data1, *d_data2;
    cudaMalloc(&d_data1, size1);
    cudaMalloc(&d_data2, size2);

    cudaStream_t stream1, stream2;
    cudaStreamCreate(&stream1);
    cudaStreamCreate(&stream2);

    // Copy and launch kernel on stream1
    cudaMemcpyAsync(d_data1, h_data1, size1, cudaMemcpyHostToDevice, stream1);
    int threads = 256;
    int blocks1 = (n1 + threads - 1) / threads;
    kernel1<<<blocks1, threads, 0, stream1>>>(d_data1, n1);

    // Copy and launch kernel on stream2
    cudaMemcpyAsync(d_data2, h_data2, size2, cudaMemcpyHostToDevice, stream2);
    int blocks2 = (n2 + threads - 1) / threads;
    kernel2<<<blocks2, threads, 0, stream2>>>(d_data2, n2);

    // Copy results back asynchronously
    cudaMemcpyAsync(h_data1, d_data1, size1, cudaMemcpyDeviceToHost, stream1);
    cudaMemcpyAsync(h_data2, d_data2, size2, cudaMemcpyDeviceToHost, stream2);

    // Wait for streams to finish
    cudaStreamSynchronize(stream1);
    cudaStreamSynchronize(stream2);

    // Print some results
    printf("Stream 1 results:\n");
    for (int i = 0; i < 5; i++) printf("h_data1[%d] = %f\n", i, h_data1[i]);

    printf("Stream 2 results:\n");
    for (int i = 0; i < 5; i++) printf("h_data2[%d] = %f\n", i, h_data2[i]);

    // Cleanup
    cudaStreamDestroy(stream1);
    cudaStreamDestroy(stream2);
    cudaFree(d_data1);
    cudaFree(d_data2);
    free(h_data1);
    free(h_data2);

    return 0;
}
```

| API | Purpose |
| --- | --- |
| `cudaMemcpyAsync` | Starts a memory copy asynchronously and returns immediately. |
| `cudaStream_t` | Represents a CUDA stream (a queue of commands executed in order). |
| `cudaStreamCreate()` | Creates a new CUDA stream. |
| `cudaStreamSynchronize()` | Blocks host until all commands in the stream have completed. |
| `cudaStreamDestroy()` | Destroys the CUDA stream and frees associated resources. |

> **Key Points**
>
> 
>  1. Asynchronous CUDA APIs allow overlapping memory transfers and kernel execution.
>  1. CUDA streams enable concurrent execution of multiple tasks on the GPU.
>  1. Proper synchronization is crucial to ensure data integrity and correctness.
>  1. Cleanup of streams and device memory is necessary to avoid resource leaks.

## Exercise 06: Asynchronous CUDA Calls

**Time: 20 min**

Read the synchronous and asynchronous vector-add sources, then submit the job.
It runs both prebuilt programs in that order.
[Exercise instructions](../docs/source/tutorial/exercises/06-asynchronous.rst).

Run the setup cell first. Build the exercises once in
[02-gpu.ipynb](02-gpu.ipynb) before submitting jobs from this notebook.
After editing CUDA source, return to that notebook to rebuild.

Submission cells use the same `qsub` commands as the exercise documentation,
from `job-script`, and immediately show `qstat`. A submitted job may still be queued.
Use the monitoring cell to check again; rerunning a submission cell creates another job.
Read the PBS output and inspect generated files after the job finishes.

<a href="../exercises/06-asynchronous/vector-add.cu" target="_blank" rel="noopener noreferrer">Open vector-add.cu</a>

<a href="../exercises/06-asynchronous/vector-add-async.cu" target="_blank" rel="noopener noreferrer">Open vector-add-async.cu</a>

In [ ]:
# Submit a new job, then inspect the queue.
submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 06-asynchronous.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


### Check the queue again

Run this cell to monitor jobs without submitting another one. Once your job is no
longer listed, rerun its output/visualisation cells. Inspect the PBS log for errors.

In [ ]:
!qstat -u "{username}"


In [ ]:
!qstat -u "{username}"
show_job_output("06-asynchronous", job_id)


### Compare the results

| Version | Reported total time (ms) | Verification |
| --- | --- | --- |
| Synchronous | | |
| Asynchronous, four streams | | |

Read the timing boundaries before interpreting the difference. The synchronous
version includes per-iteration verification and printing; the asynchronous
version verifies after timing. Its host buffers use ordinary `malloc`, so transfer
and kernel overlap is not guaranteed. These reported times do not isolate the
benefit of streams. Repeat the same documented job to explore variability.